# EV 화재 FDS 입력 생성기 — KRISO-Fire 2차년도`KRISO_Fire_FDS_project_brief.md` 6~7장을 **파라미터 기반 생성기**로 구현한 것.`.fds`를 직접 손으로 쓰지 않고, 이 노트북의 파라미터를 바꿔서 케이스를 찍어낸다.## 왜 노트북으로 만드는가1. **브리프 8장 검증 체크리스트 5·6번**(격자 민감도, 복사분율 민감도)은 같은 입력을   파라미터만 바꿔 여러 벌 만들어야 한다. 손으로 쓰면 반드시 어긋난다.2. **1D → 3D 인터페이스**(브리프 5.1)는 CSV → `&RAMP` 변환 스크립트가 필수다   (브리프 7.1 주의사항 4번). 그 변환이 여기 들어 있다.3. `MASS_FLUX`, `THICKNESS` 같은 값은 **지오메트리에 의존**한다. 격자에 스냅된   실제 면적을 모르면 총 방출질량/총 발열량이 어긋난다. 노트북이 이를 계산한다.## 파이프라인```input/data_1d/*.csv          1D 팀(MPSE) 산출물 (지금은 더미)        │        ▼  이 노트북input/cases/<CHID>/<CHID>.fds  +  run_<CHID>.sbatch        │        ▼  Slurm (KISTI Neuron cpu 파티션)results/<CHID>/              _hrr.csv, _devc.csv, *.sf, *.bf ...        │        ▼  results/postprocess.ipynbresults/figures/```> 이 노트북이 만드는 모든 수치는 브리프의 `[PH]` 플레이스홀더 수준이다.> 실제 1D 데이터가 오면 `input/data_1d/`의 CSV만 교체하면 된다.

## 0. 경로 · 임포트

In [ ]:
from __future__ import annotations

import json
import math
import shutil
import subprocess
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

ROOT    = Path("/scratch/x3319a05/FDS/electric_vehicle_battery")
INPUT   = ROOT / "input"
DATA_1D = INPUT / "data_1d"
CASEDIR = INPUT / "cases"
RESULTS = ROOT / "results"

FDS_ENV = "/scratch/x3319a05/FDS/setup_docs/scripts/fds_env_kisti.sh"
FDS_BIN = "/scratch/x3319a05/FDS/Build/impi_intel_linux/fds_impi_intel_linux"

for d in (DATA_1D, CASEDIR, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

_trapz = getattr(np, "trapezoid", None) or np.trapz
print("data_1d :", sorted(p.name for p in DATA_1D.glob("*.csv")))

## 1. 층 1 — 성분(primitive) 물성 DB브리프 4.4의 "파라미터가 들어가는 4개 층" 중 **1층**.FDS Appendix A 등재 species는 물성 입력이 필요 없고 `LUMPED_COMPONENT_ONLY=T`만붙이면 된다. 여기 있는 `W`, `dhc`는 FDS에 넘기는 값이 아니라 **노트북이 유효분자량과 `HEAT_OF_COMBUSTION`을 유도하기 위해** 쓰는 값이다 (브리프 4.5 ①).`HYDROGEN FLUORIDE`도 FDS 6.10 Appendix A에 등재되어 있음을 확인했다(`Source/prop.f90` THERMO_DATA(218), `Source/data.f90`의 FED 계수 FLD_IRR/FIC_IRR).따라서 물성 직접 입력이 필요 없다 — 브리프 9.3 체크 항목 해소.

In [ ]:
# key : (FDS SPEC ID, 분자량 [g/mol], 연소열 [MJ/kg])
SPECIES_DB = {
    "H2":   ("HYDROGEN",         2.016, 120.0),
    "CO":   ("CARBON MONOXIDE", 28.010,  10.1),
    "CO2":  ("CARBON DIOXIDE",  44.010,   0.0),   # 불활성 — 산소요구량 0 (브리프 4.3)
    "CH4":  ("METHANE",         16.043,  50.0),
    "C2H4": ("ETHYLENE",        28.054,  47.2),
    "HF":   ("HYDROGEN FLUORIDE", 20.006, 0.0),   # 비반응 추적 (브리프 4.5 ④)
}
FDS_ID = {k: v[0] for k, v in SPECIES_DB.items()}
MW     = {k: v[1] for k, v in SPECIES_DB.items()}
DHC    = {k: v[2] for k, v in SPECIES_DB.items()}

## 2. 층 2~4 — 케이스 파라미터 정의여기가 **유일한 튜닝 지점**이다. 아래 dataclass의 기본값이 브리프 6장의플레이스홀더이며, 셀 하나만 바꿔서 케이스를 찍어낸다.

In [ ]:
@dataclass
class Grid:
    """도메인 · 균일 격자 · MPI 메쉬 분할."""
    xmin: float = 0.0;  xmax: float = 10.0
    ymin: float = 0.0;  ymax: float = 6.0
    zmin: float = 0.0;  zmax: float = 5.0
    dx: float = 0.125            # 균일 격자 [m]. 차체 형상 해상도가 여기 걸림
    # MPI 메쉬 분할 (nx,ny,nz) -> rank 수 = 곱.
    #  - z 경계면(2.5 m)은 차량 루프(1.75 m)보다 위라 차체·팩 주변 화염을 자르지 않는다.
    #  - x 2분할의 균등 경계면은 5.0 m인데, 이는 벤트 존 Z2의 정중앙이다.
    #    소스를 반으로 가르면 두 메쉬에 걸쳐 유량이 분배되므로 `cuts`로 비켜 놓는다.
    #  - KISTI cpu 파티션은 대개 포화 상태다. 8-rank는 backfill이 안 되어 하루씩
    #    대기하지만 2~4 rank는 즉시 스케줄된다.
    nmesh: tuple = (2, 1, 2)
    # 축별 명시적 분할면 [m]. None이면 셀 수 균등분할.
    cuts: tuple = (None, None, None)

    @property
    def ijk(self):
        return (round((self.xmax - self.xmin) / self.dx),
                round((self.ymax - self.ymin) / self.dx),
                round((self.zmax - self.zmin) / self.dx))

    @property
    def ncell(self):
        i, j, k = self.ijk
        return i * j * k

    @property
    def nranks(self):
        return self.nmesh[0] * self.nmesh[1] * self.nmesh[2]


@dataclass
class Vehicle:
    """차량 형상. 정육면체가 아니라 후드/캐빈/트렁크/타이어로 나눈 계단 실루엣."""
    cx: float = 5.0              # 차량 중심 x [m]
    cy: float = 3.0              # 차량 중심 y [m]
    length: float = 4.50         # 전장 [m]  (코나 EV ~4.36, IONIQ5 ~4.64)
    width:  float = 1.80         # 전폭 [m]

    # --- 수직 레벨 [m]
    z_pack_bot:  float = 0.150   # 팩 하면 (지상고)
    z_pack_top:  float = 0.400   # 팩 상면 = 플로어팬 하면
    z_floor_top: float = 0.600   # 플로어팬 상면
    z_hood:      float = 1.050   # 후드 상면
    z_belt:      float = 1.200   # 벨트라인(도어 상단) = 캐빈 하단
    z_trunk:     float = 1.150   # 트렁크 상면
    z_roof:      float = 1.750   # 루프

    # --- 길이방향 분할 (전장에 대한 비율)
    f_hood_end:    float = 0.29
    f_cabin_start: float = 0.35
    f_cabin_end:   float = 0.80
    f_rear_start:  float = 0.86
    f_pillar:      float = 0.07  # A/C 필러(윈드실드 경사) 구간 길이 비율
    cabin_inset:   float = 0.15  # 캐빈 폭 축소량 (편측) [m]

    # --- 타이어
    tire_d: float = 0.650        # 외경 [m]
    tire_w: float = 0.250        # 폭 [m]
    f_axle: float = 0.189        # 차체 끝에서 축까지 거리 / 전장 (휠베이스 ~2.6 m)

    # --- 배터리 팩 (연소하지 않는 OBST)
    pack_len: float = 2.00
    pack_wid: float = 1.40


@dataclass
class VentLayout:
    """벤트 존 배치. 팩 측면(±y)에서 방출 — 실차의 팩 상면은 플로어팬에 막혀 있다."""
    n_zone: int = 3
    zone_len: float = 0.50       # x 방향 길이 [m]
    sides: tuple = ("YMIN", "YMAX")
    # 벤트 제트는 해상하지 않는다 (브리프 6.2 경고). 실제 벤트구보다 넓은
    # 등가 면적에 분산시켜 운동량을 낮춘 것이 위 zone_len x 팩높이 x 2면.


@dataclass
class BatteryGas:
    """층 2(조성) + 층 3(REAC) + 층 4(소스)."""
    # 층 2 — 조성 (부피분율 %). 1D CSV의 질량분율에서 역산해도 되지만,
    #        브리프 6.3이 부피분율로 주어져 있어 그대로 쓴다.
    composition: dict = field(default_factory=lambda: {
        "H2": 28.0, "CO": 23.0, "CO2": 28.0, "CH4": 12.0, "C2H4": 9.0})
    # 층 3 — REAC
    soot_yield: float = 0.02
    co_yield: float = 0.05             # 조성 CO와 별개 (브리프 4.5 ②)
    radiative_fraction: float = 0.15   # 기본값 0.35 금지 (브리프 4.5 ③)
    heat_of_combustion: Optional[float] = None   # None -> 조성에서 자동 유도
    # 층 4 — 소스
    track_hf: bool = True


@dataclass
class CarBody:
    """차체 등가 액체연료 (증발 모델). 브리프 6.4 매핑표."""
    density: float = 900.0             # rho_s   [kg/m3]
    specific_heat: float = 1.5         # c_p,s   [kJ/(kg K)]
    conductivity: float = 0.20         # k_s     [W/(m K)]
    boiling_temperature: float = 350.0 # T_b     [C]  <- 이 줄이 액체 모델을 켠다
    heat_of_reaction: float = 500.0    # h_v     [kJ/kg] = 기화 잠열
    absorption_coefficient: float = 1000.0
    emissivity: float = 0.90
    formula: str = "C6H10O1"           # W       (브리프 9.3 재검토 항목)
    heat_of_combustion: float = 25000. # dH_e    [kJ/kg]
    soot_yield: float = 0.10
    co_yield: float = 0.05
    radiative_fraction: float = 0.35
    thickness_body: float = 0.012      # <- 총 THR 주 튜닝 노브 (브리프 6.4)
    thickness_tire: float = 0.020      # 타이어는 더 두껍게


@dataclass
class Casing:
    """경로 ① — 팩 케이싱 고체 경계조건. NET_HEAT_FLUX 사용 (브리프 6.5)."""
    use_net_heat_flux: bool = True     # False면 INERT (민감도용)


@dataclass
class Case:
    chid: str
    title: str
    t_end: float = 900.0
    tmpa: float = 20.0                 # 주변 온도 [C]
    grid: Grid = field(default_factory=Grid)
    veh: Vehicle = field(default_factory=Vehicle)
    vent: VentLayout = field(default_factory=VentLayout)
    gas: BatteryGas = field(default_factory=BatteryGas)
    body: CarBody = field(default_factory=CarBody)
    casing: Casing = field(default_factory=Casing)
    ramp_eps: float = 0.004            # RAMP 데이터 감축 허용오차 (F 단위)
    dt_hrr: float = 1.0
    dt_devc: float = 1.0
    dt_slcf: float = 2.0
    dt_bndf: float = 10.0
    walltime: str = "08:00:00"

## 3. 유도 열화학 — 브리프 4.5 ① 함정 방지`HEAT_OF_COMBUSTION`은 **불활성분(CO₂)을 포함한 lumped species 1 kg당** 값이다.가연분 기준 값을 넣으면 HRR이 통째로 과대평가된다.$$W_{mix}=\sum_i x_i W_i,\qquad Y_i=\frac{x_i W_i}{W_{mix}},\qquad\Delta H_{c,mix}=\sum_i Y_i \Delta H_{c,i}$$

In [ ]:
def thermochem(gas: BatteryGas) -> dict:
    x = {k: v / sum(gas.composition.values()) for k, v in gas.composition.items()}
    w_mix = sum(x[k] * MW[k] for k in x)
    Y = {k: x[k] * MW[k] / w_mix for k in x}
    dhc_mix = sum(Y[k] * DHC[k] for k in Y) * 1000.0        # [kJ/kg]
    y_fuel = sum(Y[k] for k in Y if DHC[k] > 0)             # 가연분 질량분율
    return dict(x=x, Y=Y, W_mix=w_mix, dhc_mix=dhc_mix, y_fuel=y_fuel,
                dhc_fuel_basis=dhc_mix / y_fuel)


def show_thermochem(gas: BatteryGas):
    tc = thermochem(gas)
    df = pd.DataFrame({
        "FDS SPEC ID": [FDS_ID[k] for k in tc["x"]],
        "부피분율 x_i": [tc["x"][k] for k in tc["x"]],
        "분자량 W_i [g/mol]": [MW[k] for k in tc["x"]],
        "질량분율 Y_i": [tc["Y"][k] for k in tc["x"]],
        "dHc,i [MJ/kg]": [DHC[k] for k in tc["x"]],
        "기여 Y_i*dHc,i [MJ/kg]": [tc["Y"][k] * DHC[k] for k in tc["x"]],
    }, index=list(tc["x"]))
    print(df.to_string(float_format=lambda v: f"{v:9.4f}"))
    print()
    print(f"  유효 분자량 W_mix          = {tc['W_mix']:8.3f} g/mol")
    print(f"  가연분 질량분율            = {tc['y_fuel']:8.4f}")
    print(f"  HEAT_OF_COMBUSTION (채택)  = {tc['dhc_mix']:8.0f} kJ/kg   <- 불활성 포함 기준")
    print(f"  (가연분 기준 환산값)       = {tc['dhc_fuel_basis']:8.0f} kJ/kg   <- 이 값을 넣으면 HRR 과대")
    return tc


TC = show_thermochem(BatteryGas())

## 4. 1D → 3D 인터페이스 (브리프 5.1)`data_1d/`의 CSV를 읽어 `&RAMP`로 변환한다. FDS `&RAMP`는 점 사이를 선형보간하므로1D 데이터 점을 그대로 넣으면 되지만, Δt=0.5 s × 900 s × 3존 = 5400줄이 되므로**Ramer–Douglas–Peucker 로 선형보간 오차 `ramp_eps` 이내에서 점을 솎아낸다.**사다리꼴 램프는 몇 점으로 줄고, 실제 1D의 들쭉날쭉한 이력은 필요한 만큼만 남는다.

In [ ]:
def rdp_mask(t, f, eps):
    """선형보간 오차가 eps 이내가 되도록 남길 점의 불리언 마스크."""
    t = np.asarray(t, float); f = np.asarray(f, float)
    keep = np.zeros(len(t), bool)
    keep[0] = keep[-1] = True
    stack = [(0, len(t) - 1)]
    while stack:
        i, j = stack.pop()
        if j <= i + 1:
            continue
        seg = np.interp(t[i:j + 1], [t[i], t[j]], [f[i], f[j]])
        d = np.abs(f[i:j + 1] - seg)
        k = int(np.argmax(d))
        if d[k] > eps:
            k += i
            keep[k] = True
            stack += [(i, k), (k, j)]
    return keep


def make_ramp(ramp_id, t, f, eps):
    """정규화된 f(t) -> &RAMP 줄 목록."""
    m = rdp_mask(t, f, eps)
    return [f"&RAMP ID='{ramp_id}', T={t[i]:8.2f}, F={f[i]:7.5f} /" for i in np.flatnonzero(m)]


def load_1d(n_zone=3):
    """1D 팀 CSV 로드. 없으면 더미 생성기를 돌린다."""
    if not (DATA_1D / "vent_zone1.csv").exists():
        subprocess.run(["python3", str(DATA_1D / "make_dummy_1d_data.py")], check=True)
    zones = [pd.read_csv(DATA_1D / f"vent_zone{i+1}.csv") for i in range(n_zone)]
    casing = pd.read_csv(DATA_1D / "casing_heat_flux.csv")
    thr = pd.read_csv(DATA_1D / "thr_1d.csv")
    return zones, casing, thr


ZONES_1D, CASING_1D, THR_1D = load_1d()

_rows = []
for i, z in enumerate(ZONES_1D, 1):
    _rows.append(dict(zone=f"Z{i}",
                      dt_s=float(np.diff(z.t_s).max()),
                      t_open=float(z.t_s[z.mdot_kg_s > 0].min()),
                      t_close=float(z.t_s[z.mdot_kg_s > 0].max()),
                      mdot_peak=float(z.mdot_kg_s.max()),
                      mass_kg=float(_trapz(z.mdot_kg_s, z.t_s)),
                      T_vent_peak_C=float(z.T_vent_K.max() - 273.15),
                      Y_HF=float(z.Y_HF.max())))
DF_1D = pd.DataFrame(_rows)
print(DF_1D.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
assert DF_1D.dt_s.max() <= 1.0, "브리프 5.1 요구조건 위반: 1D 데이터 Δt > 1 s"
print(f"\n1D 총 방출질량 {DF_1D.mass_kg.sum():.2f} kg, "
      f"1D THR {THR_1D.THR_MJ.iloc[-1]:.1f} MJ, "
      f"케이싱 피크 {CASING_1D.q_net_kW_m2.max():.2f} kW/m2")

## 5. 지오메트리 빌더브리프 6.2를 **차량 형상**으로 확장했다. 정육면체 하나가 아니라```                     ___________                ____/  캐빈     \____              z=1.75  루프      _________/                     \____        z=1.20  벨트라인     /  후드                            트렁크\    z=1.05 / 1.15    |________________도어_____________________|   z=0.60  플로어팬 상면    |__________________________________________|  z=0.40  플로어팬 하면        (O)      [ 배터리 팩 ]          (O)        z=0.15~0.40 팩   ────────────────────────────────────────────   z=0  지면```### 벤트를 팩 **측면**에 두는 이유브리프 6.2의 스켈레톤은 팩 상면(z=0.40)에 `&VENT`를 두는데, 같은 좌표에 차체하면이 붙어 있어 **가스가 나갈 곳이 없다**(FDS에서 solid-solid 접촉면은 가스와접하지 않는 면이 되어 유량이 사라진다). 실차의 팩 상면도 플로어팬에 볼트로직결되어 있으므로 물리적으로도 상면 방출은 맞지 않다.여기서는 **팩 ±y 측면**에 벤트 패치를 두었다. 가스가 차량 하부에서 옆으로 뿜어나와 사이드실·타이어 주변에서 상승·착화하는데, 이는 Kang et al. 시험에서 벤트제트가 타이어를 태운 거동(브리프 6.2)과 직접 대응한다.### 격자 스냅FDS는 `OBST`/`VENT` 좌표를 격자에 스냅한다. 스냅 **후**의 면적으로 `MASS_FLUX`를계산해야 총 방출질량이 1D와 일치한다. `snap_box()`가 이를 처리하고, 두께가0셀이 되는 부재는 최소 1셀로 강제한다.

In [ ]:
def snap_box(xb, g: Grid, min_cells=1):
    """OBST/VENT 좌표를 격자에 스냅. 평면(두께 0)은 평면으로 유지."""
    o = (g.xmin, g.ymin, g.zmin)
    out = []
    for a in range(3):
        lo, hi = xb[2 * a], xb[2 * a + 1]
        i0 = round((lo - o[a]) / g.dx)
        i1 = round((hi - o[a]) / g.dx)
        if abs(hi - lo) < 1e-12:
            i1 = i0                       # 평면 유지
        elif i1 - i0 < min_cells:
            i1 = i0 + min_cells           # 최소 두께 강제
        out += [o[a] + i0 * g.dx, o[a] + i1 * g.dx]
    return tuple(round(v, 6) for v in out)


def build_geometry(c: Case):
    """OBST / VENT 목록을 만든다. 반환값은 전부 격자에 스냅된 좌표."""
    v, g = c.veh, c.grid
    L, W = v.length, v.width
    x0, x1 = v.cx - L / 2, v.cx + L / 2
    y0, y1 = v.cy - W / 2, v.cy + W / 2
    xh  = x0 + v.f_hood_end * L
    xc0 = x0 + v.f_cabin_start * L
    xc1 = x0 + v.f_cabin_end * L
    xr  = x0 + v.f_rear_start * L
    dp  = v.f_pillar * L
    yi0, yi1 = y0 + v.cabin_inset, y1 - v.cabin_inset
    z_mid = 0.5 * (v.z_belt + v.z_roof)

    obst = []   # (name, xb, surf_id, color)

    # --- 차체 (가연물)
    obst.append(("FLOORPAN", (x0, x1, y0, y1, v.z_pack_top, v.z_floor_top), "CAR_BODY", "GRAY 60"))
    obst.append(("HOOD",     (x0, xh, y0, y1, v.z_floor_top, v.z_hood),     "CAR_BODY", "SILVER"))
    obst.append(("DOORS",    (xh, xr, y0, y1, v.z_floor_top, v.z_belt),     "CAR_BODY", "SILVER"))
    obst.append(("TRUNK",    (xr, x1, y0, y1, v.z_floor_top, v.z_trunk),    "CAR_BODY", "SILVER"))
    # 캐빈: 윈드실드(A필러) / 루프 / 백라이트(C필러) 3단으로 경사 근사
    obst.append(("WINDSHIELD", (xc0, xc0 + dp, yi0, yi1, v.z_belt, z_mid),      "CAR_BODY", "SKY BLUE"))
    obst.append(("ROOF",       (xc0 + dp, xc1 - dp, yi0, yi1, v.z_belt, v.z_roof), "CAR_BODY", "SILVER"))
    obst.append(("BACKLIGHT",  (xc1 - dp, xc1, yi0, yi1, v.z_belt, z_mid),      "CAR_BODY", "SKY BLUE"))

    # --- 타이어 (가연물, 두께만 다름)
    xa_f = x0 + v.f_axle * L
    xa_r = x1 - v.f_axle * L
    for tag, xa in (("F", xa_f), ("R", xa_r)):
        for side, ya, yb in (("L", y0 - 0.05, y0 + v.tire_w - 0.05),
                             ("R", y1 - v.tire_w + 0.05, y1 + 0.05)):
            obst.append((f"TIRE_{tag}{side}",
                         (xa - v.tire_d / 2, xa + v.tire_d / 2, ya, yb, 0.0, v.tire_d),
                         "TIRE", "BLACK"))

    # --- 배터리 팩 (비연소)
    px0, px1 = v.cx - v.pack_len / 2, v.cx + v.pack_len / 2
    py0, py1 = v.cy - v.pack_wid / 2, v.cy + v.pack_wid / 2
    pack = ("PACK", (px0, px1, py0, py1, v.z_pack_bot, v.z_pack_top), "PACK_CASE", "GRAY 30")

    obst = [(n, snap_box(xb, g), s, col) for n, xb, s, col in obst]
    pack = (pack[0], snap_box(pack[1], g), pack[2], pack[3])

    # --- 벤트 패치 (팩 ±y 측면). 팩 스냅 좌표 기준으로 생성.
    P = pack[1]
    vents = []
    for i in range(c.vent.n_zone):
        xc = P[0] + (P[1] - P[0]) * (i + 0.5) / c.vent.n_zone
        vx0, vx1 = xc - c.vent.zone_len / 2, xc + c.vent.zone_len / 2
        patches = []
        if "YMIN" in c.vent.sides:
            patches.append(snap_box((vx0, vx1, P[2], P[2], P[4], P[5]), g))
        if "YMAX" in c.vent.sides:
            patches.append(snap_box((vx0, vx1, P[3], P[3], P[4], P[5]), g))
        area = sum(abs(p[1] - p[0]) * abs(p[5] - p[4]) for p in patches)
        vents.append(dict(name=f"Z{i+1}", patches=patches, area=area))

    return dict(obst=obst, pack=pack, vents=vents,
                bbox=dict(x=(x0, x1), y=(y0, y1), z=(0.0, v.z_roof)))


def fmt_xb(xb):
    return ",".join(f"{v:.4f}" for v in xb)

## 6. 격자 래스터화 QA`OBST`를 격자에 채워 넣고 **가스와 접한 면만** 세어 실제 노출 연소면적을 구한다.FDS가 내부적으로 하는 것과 같은 계산이라, 브리프 6.4의> 연료 총량 = ρ × THICKNESS × 면적 … 노출 연소면적 ≈ 25 m²를 추정이 아니라 실측으로 대체할 수 있다. `THICKNESS`(총 THR 주 튜닝 노브)를목표 THR에 맞추는 데 그대로 쓴다.

In [ ]:
def rasterize(boxes, g: Grid):
    nx, ny, nz = g.ijk
    m = np.zeros((nx, ny, nz), bool)
    for xb in boxes:
        i0 = max(0,  round((xb[0] - g.xmin) / g.dx)); i1 = min(nx, round((xb[1] - g.xmin) / g.dx))
        j0 = max(0,  round((xb[2] - g.ymin) / g.dx)); j1 = min(ny, round((xb[3] - g.ymin) / g.dx))
        k0 = max(0,  round((xb[4] - g.zmin) / g.dx)); k1 = min(nz, round((xb[5] - g.zmin) / g.dx))
        m[i0:i1, j0:j1, k0:k1] = True
    return m


def exposed_area(mask, solid, g: Grid, ground=True):
    """mask 셀 중 가스와 접한 면의 총 면적 [m2]. 지면(z=zmin)은 solid 취급."""
    p = np.zeros(np.array(solid.shape) + 2, bool)
    p[1:-1, 1:-1, 1:-1] = solid
    if ground:
        p[:, :, 0] = True
    tot = 0
    for ax, off in ((0, -1), (0, 1), (1, -1), (1, 1), (2, -1), (2, 1)):
        sl = [slice(1, -1)] * 3
        sl[ax] = slice(1 + off, p.shape[ax] - 1 + off)
        tot += int(np.count_nonzero(mask & ~p[tuple(sl)]))
    return tot * g.dx ** 2


def geometry_qa(c: Case, geo=None, verbose=True):
    geo = geo or build_geometry(c)
    g = c.grid
    body_boxes = [xb for n, xb, s, _ in geo["obst"] if s == "CAR_BODY"]
    tire_boxes = [xb for n, xb, s, _ in geo["obst"] if s == "TIRE"]
    pack_box   = [geo["pack"][1]]

    m_body = rasterize(body_boxes, g)
    m_tire = rasterize(tire_boxes, g) & ~m_body       # 중복 셀은 차체 소유로
    m_pack = rasterize(pack_box, g) & ~(m_body | m_tire)
    solid  = m_body | m_tire | m_pack

    a_body = exposed_area(m_body, solid, g)
    a_tire = exposed_area(m_tire, solid, g)
    a_pack = exposed_area(m_pack, solid, g)

    b = c.body
    fuel_kg = b.density * (a_body * b.thickness_body + a_tire * b.thickness_tire)
    thr_body_GJ = fuel_kg * b.heat_of_combustion / 1e6
    thr_batt_MJ = DF_1D.mass_kg.sum() * thermochem(c.gas)["dhc_mix"] / 1e3

    # D* / dx  (브리프 6.1)
    q_peak_kW = 7000.0
    dstar = (q_peak_kW / (1.204 * 1.005 * 293.0 * math.sqrt(9.81))) ** 0.4

    out = dict(a_body=a_body, a_tire=a_tire, a_pack=a_pack,
               fuel_kg=fuel_kg, thr_body_GJ=thr_body_GJ, thr_batt_MJ=thr_batt_MJ,
               dstar=dstar, dstar_over_dx=dstar / g.dx,
               vent_area=sum(v["area"] for v in geo["vents"]))
    if verbose:
        i, j, k = g.ijk
        print(f"[{c.chid}]")
        print(f"  격자           : {i} x {j} x {k} = {g.ncell:,} cells, dx = {g.dx} m")
        print(f"  MPI 메쉬       : {g.nmesh} -> {g.nranks} rank, "
              f"{g.ncell // g.nranks:,} cells/rank")
        print(f"  해석시간       : {c.t_end:.0f} s")
        print(f"  D* (7 MW)      : {dstar:.2f} m,  D*/dx = {dstar/g.dx:.1f}  (권장 10~16)")
        print(f"  노출면적 차체  : {a_body:7.2f} m2   (브리프 추정 25 m2)")
        print(f"  노출면적 타이어: {a_tire:7.2f} m2")
        print(f"  노출면적 팩    : {a_pack:7.2f} m2   <- 경로 ① NET_HEAT_FLUX 인가면")
        per_zone = ", ".join("{}={:.3f}".format(z["name"], z["area"]) for z in geo["vents"])
        print(f"  벤트 면적 합   : {out['vent_area']:7.3f} m2  ({per_zone})")
        print(f"  차체 연료질량  : {fuel_kg:7.1f} kg  (THICKNESS 차체 {b.thickness_body} / 타이어 {b.thickness_tire} m)")
        print(f"  차체 기여 THR  : {thr_body_GJ:7.2f} GJ  (Kang et al. BEV 기준 7~8 GJ)")
        print(f"  배터리 기여 THR: {thr_batt_MJ:7.1f} MJ  ({thr_batt_MJ/1000/max(thr_body_GJ,1e-9)*100:.1f} % of 차체)")
        print(f"  케이싱 피크 열량: {a_pack * CASING_1D.q_net_kW_m2.max():.1f} kW (면적 x 피크 유속)")
    return out

### 6.1 형상 미리보기Smokeview를 띄우지 않고 차량 실루엣이 의도대로 나왔는지 확인한다.격자에 스냅된 **실제 셀**을 그리므로, 격자를 거칠게 하면 여기서 바로 뭉개지는 게 보인다.(이 클러스터에는 한글 폰트가 없어 그림 안 글자는 영문으로 쓴다.)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PART_COLOR = {"CAR_BODY": "#b9c2cc", "TIRE": "#2b2b2b", "PACK_CASE": "#c8792b"}


def preview_geometry(c: Case, save=None):
    geo, g, v = build_geometry(c), c.grid, c.veh
    groups = {"CAR_BODY": [], "TIRE": [], "PACK_CASE": []}
    for _, xb, surf, _ in geo["obst"]:
        groups[surf].append(xb)
    groups["PACK_CASE"].append(geo["pack"][1])
    masks = {k: rasterize(b, g) for k, b in groups.items() if b}

    x = np.linspace(g.xmin, g.xmax, g.ijk[0] + 1)
    y = np.linspace(g.ymin, g.ymax, g.ijk[1] + 1)
    z = np.linspace(g.zmin, g.zmax, g.ijk[2] + 1)

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    views = [(0, "side  (x-z, y=centre)", x, z, lambda m: m[:, g.ijk[1] // 2, :], "x [m]", "z [m]"),
             (1, "top   (x-y, z=roof..0)", x, y, lambda m: m.any(axis=2),          "x [m]", "y [m]"),
             (2, "front (y-z, x=centre)", y, z, lambda m: m[g.ijk[0] // 2, :, :],  "y [m]", "z [m]")]
    for i, title, a, b, sel, xl, yl in views:
        for k, m in masks.items():
            f = np.where(sel(m), 1.0, np.nan)
            ax[i].pcolormesh(a, b, f.T, cmap=matplotlib.colors.ListedColormap([PART_COLOR[k]]),
                             vmin=0, vmax=1, shading="flat")
        ax[i].set_title(title, fontsize=9)
        ax[i].set_xlabel(xl); ax[i].set_ylabel(yl)
        ax[i].set_aspect("equal"); ax[i].grid(alpha=0.25, lw=0.4)

    # 벤트 패치 위치 표시
    for vz in geo["vents"]:
        for p in vz["patches"]:
            ax[0].plot([p[0], p[1]], [p[4], p[4]], color="red", lw=2.5)
            ax[0].plot([p[0], p[1]], [p[5], p[5]], color="red", lw=2.5)
            ax[1].plot([p[0], p[1]], [p[2], p[3]], color="red", lw=2.5)
            ax[2].plot([p[2], p[3]], [p[4], p[5]], color="red", lw=2.5)
    ax[0].set_xlim(v.cx - 3.2, v.cx + 3.2); ax[0].set_ylim(0, v.z_roof + 0.7)
    ax[2].set_xlim(v.cy - 2.2, v.cy + 2.2); ax[2].set_ylim(0, v.z_roof + 0.7)
    fig.suptitle(f"{c.chid}  dx={g.dx} m   grey=CAR_BODY  black=TIRE  orange=PACK  red=VENT patch",
                 fontsize=9)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=140, bbox_inches="tight")
    return fig


preview_geometry(Case(chid="preview", title="preview"))

## 7. FDS 입력 파일 생성기

In [ ]:
HDR = lambda s: f"\n! {'='*72}\n!  {s}\n! {'='*72}"


def snap_scalar(v, g: Grid, axis=0):
    o = (g.xmin, g.ymin, g.zmin)[axis]
    return round(o + round((v - o) / g.dx) * g.dx, 6)


def mesh_lines(g: Grid):
    """MPI 메쉬 분할. g.cuts가 주어진 축은 그 면에서, 아니면 셀 수 균등분할."""
    ijk = g.ijk
    lo = (g.xmin, g.ymin, g.zmin)

    def split(n_cell, n_mesh, o, explicit, axis):
        if explicit:
            bnd = sorted({0, n_cell} | {round((snap_scalar(v, g, axis) - o) / g.dx) for v in explicit})
            if len(bnd) - 1 != n_mesh:
                raise ValueError(f"cuts[{axis}]={explicit} -> {len(bnd)-1} 메쉬, nmesh={n_mesh}와 불일치")
        else:
            base, rem = divmod(n_cell, n_mesh)
            bnd, acc = [0], 0
            for m in range(n_mesh):
                acc += base + (1 if m < rem else 0)
                bnd.append(acc)
        return [(bnd[i + 1] - bnd[i], o + bnd[i] * g.dx, o + bnd[i + 1] * g.dx)
                for i in range(len(bnd) - 1)]

    cx = split(ijk[0], g.nmesh[0], lo[0], g.cuts[0], 0)
    cy = split(ijk[1], g.nmesh[1], lo[1], g.cuts[1], 1)
    cz = split(ijk[2], g.nmesh[2], lo[2], g.cuts[2], 2)
    lines, n = [], 0
    for a in cx:
        for b in cy:
            for cc in cz:
                n += 1
                lines.append(f"&MESH ID='M{n:02d}', IJK={a[0]},{b[0]},{cc[0]}, "
                             f"XB={a[1]:.4f},{a[2]:.4f}, {b[1]:.4f},{b[2]:.4f}, "
                             f"{cc[1]:.4f},{cc[2]:.4f} /")
    planes = dict(x=[a[2] for a in cx[:-1]], y=[b[2] for b in cy[:-1]], z=[c[2] for c in cz[:-1]])
    return lines, planes


def write_fds(c: Case, outdir: Path) -> Path:
    g, v, b = c.grid, c.veh, c.body
    geo = build_geometry(c)
    tc = thermochem(c.gas)
    hoc = c.gas.heat_of_combustion or tc["dhc_mix"]
    outdir.mkdir(parents=True, exist_ok=True)
    L = []

    L.append(f"&HEAD CHID='{c.chid}', TITLE='{c.title}' /")
    L.append("! 생성: input/build_fds_input.ipynb  (수동 편집 금지 — 노트북을 고칠 것)")
    L.append(f"! 1D 소스: input/data_1d/  (Δt={DF_1D.dt_s.max():.2f} s)")

    L.append(HDR("메쉬 · 시간 · 출력"))
    ml, planes = mesh_lines(g)
    L += ml
    L.append(f"! 메쉬 경계면: x={planes['x']}, y={planes['y']}, z={planes['z']}")
    L.append(f"&TIME T_END={c.t_end:.1f} /")
    L.append(f"&MISC TMPA={c.tmpa:.1f} /")
    L.append(f"&DUMP DT_HRR={c.dt_hrr}, DT_DEVC={c.dt_devc}, DT_SLCF={c.dt_slcf}, "
             f"DT_BNDF={c.dt_bndf}, MASS_FILE=T /")

    # ---------------- 화학종 (층 1~2)
    L.append(HDR("화학종  (브리프 4장: 층 1 성분 -> 층 2 lumped)"))
    for k in c.gas.composition:
        L.append(f"&SPEC ID={chr(39)+FDS_ID[k]+chr(39):<19} LUMPED_COMPONENT_ONLY=T /")
    L.append("")
    L.append("! 배터리 벤트가스 lumped fuel. CO2를 안에 포함 (브리프 4.3)")
    L.append("!   -> 원자 balance상 CO2의 산소요구량은 0이므로 그대로 넣어도 무방하고,")
    L.append("!      빼면 질량유량이 틀리고 희석·질식 효과가 사라진다.")
    parts = [f"&SPEC ID='BATTERY_GAS'"]
    for n, k in enumerate(c.gas.composition, 1):
        parts.append(f"      SPEC_ID({n})='{FDS_ID[k]}', VOLUME_FRACTION({n})={c.gas.composition[k]:.3f}")
    L.append(",\n".join(parts) + " /")
    L.append("")
    L.append(f"&SPEC ID='CAR_BODY_FUEL', FORMULA='{b.formula}' /")
    if c.gas.track_hf:
        L.append("! HF: lumped fuel에 넣을 수 없다 (C,H,N,O 원자만 허용, 브리프 4.5 ④).")
        L.append("!     비반응 추적 species로 벤트에서 동시 주입한다.")
        L.append(f"&SPEC ID='{FDS_ID['HF']}' /")

    # ---------------- 반응 (층 3)
    L.append(HDR("반응  (층 3: 연료 2종 -> REAC 2줄)"))
    L.append(f"&REAC ID='BATT', FUEL='BATTERY_GAS',")
    L.append(f"      HEAT_OF_COMBUSTION={hoc:.1f},   ! 불활성(CO2) 포함 기준 (브리프 4.5 ①)")
    L.append(f"      SOOT_YIELD={c.gas.soot_yield}, CO_YIELD={c.gas.co_yield},  ! 조성 CO와 별개 (4.5 ②)")
    L.append(f"      RADIATIVE_FRACTION={c.gas.radiative_fraction} /   ! 기본 0.35 금지, H2 우세 화염 (4.5 ③)")
    L.append("")
    L.append(f"&REAC ID='BODY', FUEL='CAR_BODY_FUEL',")
    L.append(f"      HEAT_OF_COMBUSTION={b.heat_of_combustion:.1f},")
    L.append(f"      SOOT_YIELD={b.soot_yield}, CO_YIELD={b.co_yield},")
    L.append(f"      RADIATIVE_FRACTION={b.radiative_fraction} /")

    # ---------------- 재료 · 표면 (층 4)
    L.append(HDR("차체 재료 — 증발(액체) 모델  (브리프 3.2, 6.4)"))
    L.append("! BOILING_TEMPERATURE가 있으면 FDS가 액체 열분해 모델을 쓰고 N_REACTIONS=1이")
    L.append("! 자동 설정되어, 유일한 '반응'이 액체->기체 상변화가 된다.")
    L.append("! 따라서 HEAT_OF_REACTION은 기화 잠열 h_v로 해석된다.")
    L.append(f"""&MATL ID='CAR_BODY_LIQUID'
      SPEC_ID='CAR_BODY_FUEL'
      NU_SPEC=1.0
      BOILING_TEMPERATURE={b.boiling_temperature:.1f}
      HEAT_OF_REACTION={b.heat_of_reaction:.1f}
      DENSITY={b.density:.1f}
      CONDUCTIVITY={b.conductivity}
      SPECIFIC_HEAT={b.specific_heat}
      ABSORPTION_COEFFICIENT={b.absorption_coefficient:.1f}
      EMISSIVITY={b.emissivity} /""")
    L.append("")
    L.append(f"&SURF ID='CAR_BODY', MATL_ID='CAR_BODY_LIQUID', THICKNESS={b.thickness_body}, COLOR='SILVER' /")
    L.append(f"&SURF ID='TIRE',     MATL_ID='CAR_BODY_LIQUID', THICKNESS={b.thickness_tire}, COLOR='BLACK' /")
    L.append("&SURF ID='GROUND', COLOR='GRAY 40' /   ! 비활성 지면")

    # ---------------- 경로 ① 케이싱
    L.append(HDR("경로 ① — 팩 케이싱 고체 경계조건  (브리프 3.1, 6.5)"))
    q_peak = float(CASING_1D.q_net_kW_m2.max())
    if c.casing.use_net_heat_flux and q_peak > 0:
        L.append("! TMP_FRONT(온도지정)이 아니라 NET_HEAT_FLUX(열유속지정)를 쓴다:")
        L.append("!  - 온도를 고정하면 FDS가 계산한 화염 복사가 팩 표면에 영향을 못 준다")
        L.append("!  - 양방향 커플링으로 확장할 때 열유속 방식이 그대로 이어진다")
        L.append("! FDS 규약: NET_HEAT_FLUX > 0 이면 벽이 주변 기체를 가열한다.")
        L.append(f"&SURF ID='PACK_CASE', COLOR='GRAY 30', NET_HEAT_FLUX={q_peak:.4f}, RAMP_Q='cas_q' /")
        L.append("")
        L += make_ramp("cas_q", CASING_1D.t_s.values,
                       (CASING_1D.q_net_kW_m2 / q_peak).values, c.ramp_eps)
    else:
        L.append("&SURF ID='PACK_CASE', COLOR='GRAY 30' /   ! 경로 ① OFF (민감도 케이스)")

    # ---------------- 경로 ② 벤트가스
    L.append(HDR("경로 ② — 벤트가스 기체 연료 소스  (브리프 3.1, 6.3)"))
    L.append("! MASS_FLUX는 RAMP F=1.0에 해당하는 기준값. 실제 유량 = MASS_FLUX x F(t).")
    L.append("! 스냅된 실제 벤트 면적으로 나눠 총 방출질량이 1D와 일치하도록 했다.")
    L.append("! 벤트 제트는 해상하지 않는다: 실제 100 m/s급 제트를 넓은 등가 면적에")
    L.append("! 분산시켜 운동량을 낮춘 것 (브리프 6.2).")
    for i, vz in enumerate(geo["vents"]):
        z1 = ZONES_1D[i]
        mdot_pk = float(z1.mdot_kg_s.max())
        y_hf = float(z1.Y_HF.max()) if c.gas.track_hf else 0.0
        mf_gas = mdot_pk * (1 - y_hf) / vz["area"]
        mf_hf  = mdot_pk * y_hf / vz["area"]
        tpk = float(z1.T_vent_K.max() - 273.15)
        rid = f"mf_{vz['name'].lower()}"
        tid = f"tv_{vz['name'].lower()}"
        L.append("")
        L.append(f"! --- {vz['name']}: 면적 {vz['area']:.4f} m2, 피크 {mdot_pk:.5f} kg/s, "
                 f"방출질량 {_trapz(z1.mdot_kg_s, z1.t_s):.2f} kg")
        s = [f"&SURF ID='VENT_{vz['name']}', COLOR='ORANGE'",
             f"      SPEC_ID(1)='BATTERY_GAS', MASS_FLUX(1)={mf_gas:.6f}, RAMP_MF(1)='{rid}'"]
        if c.gas.track_hf:
            s.append(f"      SPEC_ID(2)='{FDS_ID['HF']}', MASS_FLUX(2)={mf_hf:.6f}, RAMP_MF(2)='{rid}'")
        s.append(f"      TMP_FRONT={tpk:.1f}, RAMP_T='{tid}'")
        L.append(",\n".join(s) + " /")
        L += make_ramp(rid, z1.t_s.values, (z1.mdot_kg_s / mdot_pk).values, c.ramp_eps)
        # TMP_FRONT 램프: TMP_F = TMPA + F*(TMP_FRONT - TMPA)
        f_t = ((z1.T_vent_K - 273.15) - c.tmpa) / (tpk - c.tmpa)
        L += make_ramp(tid, z1.t_s.values, f_t.clip(0, 1).values, c.ramp_eps)

    # ---------------- 지오메트리
    L.append(HDR("지오메트리 — 차량 형상 + 배터리 팩"))
    for n, xb, surf, col in geo["obst"]:
        L.append(f"&OBST ID='{n}', XB={fmt_xb(xb)}, SURF_ID='{surf}', COLOR='{col}' /")
    L.append("")
    L.append(f"&OBST ID='{geo['pack'][0]}', XB={fmt_xb(geo['pack'][1])}, "
             f"SURF_ID='{geo['pack'][2]}', COLOR='{geo['pack'][3]}' /   ! 연소하지 않음")
    L.append("")
    L.append("! 벤트 패치 — 팩 ±y 측면 (상면은 플로어팬에 막혀 유량이 사라진다)")
    for vz in geo["vents"]:
        for p in vz["patches"]:
            L.append(f"&VENT XB={fmt_xb(p)}, SURF_ID='VENT_{vz['name']}' /")

    L.append("")
    L.append("! 바닥 제외 전면 OPEN (브리프 6.1)")
    for mb in ("XMIN", "XMAX", "YMIN", "YMAX", "ZMAX"):
        L.append(f"&VENT MB='{mb}', SURF_ID='OPEN' /")
    L.append("&VENT MB='ZMIN', SURF_ID='GROUND' /")

    # ---------------- 출력
    P = geo["pack"][1]
    L.append(HDR("출력"))
    L.append("! --- 반응별 HRR 분리: 배터리 기여 vs 차체 기여 (브리프 8장 #1)")
    L.append(f"&DEVC ID='HRR_BATT', QUANTITY='HRRPUV REAC', REAC_ID='BATT', "
             f"SPATIAL_STATISTIC='VOLUME INTEGRAL', XB={fmt_xb((g.xmin,g.xmax,g.ymin,g.ymax,g.zmin,g.zmax))} /")
    L.append(f"&DEVC ID='HRR_BODY', QUANTITY='HRRPUV REAC', REAC_ID='BODY', "
             f"SPATIAL_STATISTIC='VOLUME INTEGRAL', XB={fmt_xb((g.xmin,g.xmax,g.ymin,g.ymax,g.zmin,g.zmax))} /")
    L.append("")
    L.append("! --- 팩 표면 입사 열유속: 1D 되먹임용 (브리프 5.2 3단계)")
    eps = g.dx * 0.01
    faces = [("PACK_BOT",  (P[0], P[1], P[2], P[3], P[4], P[4]), -3),
             ("PACK_YMIN", (P[0], P[1], P[2], P[2], P[4], P[5]), -2),
             ("PACK_YMAX", (P[0], P[1], P[3], P[3], P[4], P[5]),  2),
             ("PACK_XMIN", (P[0], P[0], P[2], P[3], P[4], P[5]), -1),
             ("PACK_XMAX", (P[1], P[1], P[2], P[3], P[4], P[5]),  1)]
    L.append("!     q_* [kW/m2] = 면 평균 (균일격자이므로 면적평균과 동일).")
    L.append("!                   1D가 가정한 경계조건과 직접 비교할 값.")
    L.append("!     Q_* [kW]     = 면적적분 (에너지 수지 확인용)")
    for nm, xb, ior in faces:
        L.append(f"&DEVC ID='q_{nm}', QUANTITY='GAUGE HEAT FLUX', XB={fmt_xb(xb)}, IOR={ior}, "
                 f"SPATIAL_STATISTIC='MEAN' /")
        L.append(f"&DEVC ID='Q_{nm}', QUANTITY='GAUGE HEAT FLUX', XB={fmt_xb(xb)}, IOR={ior}, "
                 f"SPATIAL_STATISTIC='SURFACE INTEGRAL' /")
        L.append(f"&DEVC ID='T_{nm}', QUANTITY='WALL TEMPERATURE', XB={fmt_xb(xb)}, IOR={ior}, "
                 f"SPATIAL_STATISTIC='MEAN' /")
    L.append("")
    L.append("! --- 기상 계측점 (차량 상방 플룸 중심선)")
    for z in (2.0, 3.0, 4.0):
        L.append(f"&DEVC ID='T_plume_{z:.0f}m', QUANTITY='TEMPERATURE', XYZ={v.cx:.3f},{v.cy:.3f},{z:.2f} /")
    L.append(f"&DEVC ID='O2_underbody', QUANTITY='VOLUME FRACTION', SPEC_ID='OXYGEN', "
             f"XYZ={v.cx:.3f},{v.cy:.3f},{0.5*(v.z_pack_bot+v.z_pack_top):.3f} /")
    for nm, sp, x, y, z in (("CO_1p5m", "CARBON MONOXIDE", v.cx, v.cy + 2.0, 1.5),
                            ("O2_1p5m", "OXYGEN",          v.cx, v.cy + 2.0, 1.5),
                            ("HF_1p5m", FDS_ID["HF"],      v.cx, v.cy + 2.0, 1.5)):
        if sp == FDS_ID["HF"] and not c.gas.track_hf:
            continue
        L.append(f"&DEVC ID='{nm}', QUANTITY='VOLUME FRACTION', SPEC_ID='{sp}', "
                 f"XYZ={x:.3f},{y:.3f},{z:.2f} /")
    L.append(f"&DEVC ID='BATTGAS_underbody', QUANTITY='MASS FRACTION', SPEC_ID='BATTERY_GAS', "
             f"XYZ={v.cx:.3f},{v.cy:.3f},{0.5*(v.z_pack_bot+v.z_pack_top):.3f} /   ! 미연소 축적 (8장 #7)")

    L.append("")
    L.append("! --- 경계면 출력 (팩 열유속 분포 · 차체 연소율)")
    for q in ("GAUGE HEAT FLUX", "NET HEAT FLUX", "WALL TEMPERATURE", "BURNING RATE"):
        L.append(f"&BNDF QUANTITY='{q}' /")

    L.append("")
    L.append("! --- 슬라이스")
    slc_y = [("TEMPERATURE", None), ("HRRPUV", None),
             ("VOLUME FRACTION", "OXYGEN"), ("VOLUME FRACTION", "CARBON MONOXIDE"),
             ("MASS FRACTION", "BATTERY_GAS")]
    if c.gas.track_hf:
        slc_y.append(("VOLUME FRACTION", FDS_ID["HF"]))
    # 메쉬 경계면 위의 슬라이스는 양쪽 메쉬에 중복 생성되므로 한 셀 비켜 놓는다.
    sy = v.cy - g.dx if any(abs(v.cy - pp) < 1e-9 for pp in planes["y"]) else v.cy
    sx = v.cx - g.dx if any(abs(v.cx - pp) < 1e-9 for pp in planes["x"]) else v.cx
    for q, sp in slc_y:
        sid = f", SPEC_ID='{sp}'" if sp else ""
        L.append(f"&SLCF PBY={sy:.4f}, QUANTITY='{q}'{sid} /")
    for q in ("TEMPERATURE", "HRRPUV"):
        L.append(f"&SLCF PBX={sx:.4f}, QUANTITY='{q}' /")
    L.append(f"&SLCF PBZ={0.5*(v.z_pack_bot+v.z_pack_top):.3f}, QUANTITY='TEMPERATURE' /   ! 차량 하부")
    L.append("&ISOF QUANTITY='HRRPUV', VALUE(1)=100. /")

    L.append("")
    L.append("&TAIL /")

    path = outdir / f"{c.chid}.fds"
    path.write_text("\n".join(L) + "\n", encoding="utf-8")
    return path

## 8. Slurm 배치 스크립트 생성기 (KISTI Neuron `cpu` 파티션)

In [ ]:
def write_sbatch(c: Case, outdir: Path) -> Path:
    """브리프 외 — KISTI Neuron 실행 환경. cpu 파티션, rank 수 = MESH 수."""
    n = c.grid.nranks
    txt = f"""#!/bin/bash
#SBATCH -J EV_{c.chid}
#SBATCH -p cpu
#SBATCH --nodes=1
#SBATCH --ntasks={n}
#SBATCH --cpus-per-task=1
#SBATCH --time={c.walltime}
#SBATCH --comment=etc
#SBATCH -o {RESULTS}/{c.chid}/slurm-%j.out
#SBATCH -e {RESULTS}/{c.chid}/slurm-%j.err

# 로그인 환경은 상속되지 않으므로 반드시 다시 source 한다.
source {FDS_ENV}

export OMP_NUM_THREADS=1
export I_MPI_FABRICS=shm:ofi
export I_MPI_PIN=1

RUNDIR={RESULTS}/{c.chid}
mkdir -p "$RUNDIR"
cp {CASEDIR}/{c.chid}/{c.chid}.fds "$RUNDIR"/
cd "$RUNDIR"

echo "host      : $(hostname)"
echo "ranks     : $SLURM_NTASKS  (MESH {c.grid.nmesh} = {n})"
echo "cells     : {c.grid.ncell}"
echo "start     : $(date)"

mpiexec -n "$SLURM_NTASKS" "$FDS_BIN" {c.chid}.fds

echo "end       : $(date)"
tail -n 5 {c.chid}.out
"""
    outdir.mkdir(parents=True, exist_ok=True)
    p = outdir / f"run_{c.chid}.sbatch"
    p.write_text(txt, encoding="utf-8")
    p.chmod(0o755)
    return p

## 9. 케이스 정의 및 생성- **`EV_quick`** — 파이프라인 스모크 테스트. 격자·형상은 본 케이스와 동일하고  해석시간만 짧다. 먼저 이걸 돌려서 입력이 통과하는지 확인한다.- **`EV_demo`** — 브리프 6장 기준 케이스 (900 s).- **민감도** — 브리프 8장 체크리스트 5·6번. 파일만 생성하고 실행은 선택.

In [ ]:
def x_cut_avoiding_vents(c: Case) -> float:
    """x 2분할 경계면을 벤트 존 '사이' 간격에 놓는다.

    균등분할이면 경계면이 도메인 중앙 x=5.0에 오는데 그 자리가 벤트 존 Z2의
    정중앙이다. 소스를 반으로 가르면 유량이 두 메쉬에 걸쳐 분배되고 제트가
    메쉬 인터페이스를 가로지르게 되므로, 존 사이 간격 중 도메인 중앙에 가장
    가까운 곳으로 옮긴다.
    """
    geo = build_geometry(c)
    spans = sorted((v["patches"][0][0], v["patches"][0][1]) for v in geo["vents"])
    gaps = [(spans[i][1], spans[i + 1][0]) for i in range(len(spans) - 1)]
    mid = 0.5 * (c.grid.xmin + c.grid.xmax)
    g0, g1 = min(gaps, key=lambda gp: abs(0.5 * (gp[0] + gp[1]) - mid))
    return snap_scalar(0.5 * (g0 + g1), c.grid, 0)


BASE = Case(chid="EV_demo",
            title="KRISO-Fire 1D-3D coupling example (EV pack + vehicle body)",
            t_end=900.0)
_xc = x_cut_avoiding_vents(BASE)
BASE = replace(BASE, grid=replace(BASE.grid, cuts=((_xc,), None, None)))
print(f"x 메쉬 분할면: {_xc:.4f} m  (균등분할이면 {0.5*(BASE.grid.xmin+BASE.grid.xmax):.2f} m "
      f"= 벤트 존 Z2 정중앙이라 회피)")

def _regrid(base: Case, chid, dx, title, walltime):
    """격자를 바꾸면 스냅 위치가 달라지므로 분할면도 다시 계산한다."""
    c = replace(base, chid=chid, title=title, walltime=walltime,
                grid=replace(base.grid, dx=dx, cuts=(None, None, None)))
    return replace(c, grid=replace(c.grid, cuts=((x_cut_avoiding_vents(c),), None, None)))


CASES = {
    "EV_quick": replace(BASE, chid="EV_quick", t_end=150.0, walltime="01:00:00",
                        title="EV fire example - pipeline smoke test"),
    "EV_demo":  BASE,
    # --- 브리프 8장 #5: 격자 민감도 (피크 HRR 변화 10% 이내여야 함)
    "EV_dx100": _regrid(BASE, "EV_dx100", 0.100, "Grid sensitivity dx=0.100 m", "20:00:00"),
    "EV_dx200": _regrid(BASE, "EV_dx200", 0.200, "Grid sensitivity dx=0.200 m", "04:00:00"),
    # --- 브리프 8장 #6: 복사분율 민감도 (차체 착화 시점을 지배)
    "EV_chi010": replace(BASE, chid="EV_chi010",
                         gas=replace(BASE.gas, radiative_fraction=0.10),
                         title="Radiative fraction sensitivity chi_r=0.10"),
    "EV_chi025": replace(BASE, chid="EV_chi025",
                         gas=replace(BASE.gas, radiative_fraction=0.25),
                         title="Radiative fraction sensitivity chi_r=0.25"),
}

QA = {}
for name, c in CASES.items():
    d = CASEDIR / c.chid
    f = write_fds(c, d)
    s = write_sbatch(c, d)
    QA[name] = geometry_qa(c, verbose=(name in ("EV_quick", "EV_demo")))
    preview_geometry(c, save=d / f"{c.chid}_geometry.png")
    plt.close("all")
    # 어떤 파라미터가 이 .fds를 만들었는지 케이스 폴더에 함께 남긴다.
    from dataclasses import asdict
    (d / "params.json").write_text(
        json.dumps({"case": asdict(c), "qa": QA[name]}, ensure_ascii=False,
                   indent=2, default=str), encoding="utf-8")
    print(f"  -> {f.relative_to(ROOT)}  ({f.stat().st_size/1024:.1f} kB, "
          f"{len(f.read_text().splitlines())} lines) + {s.name}\n")

In [ ]:
# 민감도 케이스 요약
pd.DataFrame(QA).T[["a_body", "a_tire", "a_pack", "vent_area",
                    "fuel_kg", "thr_body_GJ", "thr_batt_MJ", "dstar_over_dx"]]

## 10. 실행```bashcd /scratch/x3319a05/FDS/electric_vehicle_batterysbatch input/cases/EV_quick/run_EV_quick.sbatch      # 먼저 스모크 테스트sbatch input/cases/EV_demo/run_EV_demo.sbatch        # 통과하면 본 케이스```결과는 `results/<CHID>/` 에 쌓이고, `results/postprocess.ipynb` 로 브리프 8장검증 체크리스트를 확인한다.### 아직 열려 있는 것 (브리프 9장)| 항목 | 현재 처리 | 실데이터 도착 시 ||---|---|---|| 조성 시변성 | 시간불변 대표값 (4.2 옵션 a) | 변동폭 크면 `VENT_EARLY`/`VENT_LATE` 2연료로 승격 || q″_cas ↔ ṁ_vent 중복 | 1D가 분리 제공한다고 가정 | 겹치면 에너지 이중계상 — 반드시 확인 || 1D 되먹임 (5.2) | 1차만. `q_PACK_*` DEVC로 계산값 확보 | `results/postprocess.ipynb`가 CSV 내보냄 || `C6H10O1` 타당성 | 브리프 [PH] 그대로 | 실제 차량 가연물 구성비로 재산출 || THICKNESS | 노출면적 실측 기반 | Kang et al. 7~8 GJ에 맞춰 재튜닝 |